# 08 — Verification: Paper / MATLAB / Python comparison of Table 1 + derived moments

**Inputs**: all `.npz` outputs from notebooks 01–07.

**Output**: `../output/verification_table.md` — a markdown table that can be embedded directly in `verification.md` and the LaTeX paper.

**Purpose**: every row reports (a) the paper's reported value, (b) the MATLAB replication-package value (often the same; occasionally slightly different from the paper text), (c) the value our Python pipeline reproduces, and (d) the source — paper section or MATLAB filename + line range — so each cell is auditable.

Two kinds of rows:
1. **Direct calibration inputs** (e.g. $\alpha$, $\delta$, $\rho_l$, $\sigma_l$): we simply re-use the value, but we *verify* it numerically (e.g. the Markov chain's empirical 1-step autocorrelation = 0.97 to machine precision in notebook 01).
2. **Derived equilibrium quantities** ($\beta$, $K/Y$, MPC, asset-market residual): values our pipeline produces by independently solving the model.

In [1]:
from pathlib import Path
import json

import numpy as np

OUTPUT_DIR = Path('..') / 'output'
p = np.load(OUTPUT_DIR / 'params.npz')
markov = np.load(OUTPUT_DIR / 'markov.npz')
calib = np.load(OUTPUT_DIR / 'calibration.npz')
dist = np.load(OUTPUT_DIR / 'distribution_ss.npz')

# Direct values from the loaded .npz files
alpha = float(p['params__alpha'])
delta = float(p['params__delta'])
omega = float(p['params__omega'])
gamma = float(p['params__gamma'])
theta = float(p['params__theta'])
varphi = float(p['params__varphi'])
epsilon_f = float(p['params__epsilon_f'])
epsilon_w = float(p['params__epsilon_w'])
theta_nr = float(p['params__theta_nr'])
rho_l = float(p['params__rho'])
sigma_innov = float(p['params__sigma_innov'])
sigma_cross = float(markov['sigma_cross'])
y_ss = float(p['params__y_ss'])
k_ss = float(p['params__k_ss'])
k_y_ss = float(p['params__k_y_ss'])
b_agg_ss = float(p['params__b_agg_ss'])
r_ss = float(p['params__r_ss'])

beta_star = float(calib['beta_calibrated'])
A_hh = float(calib['A_hh'])
K_implied = float(calib['K_implied'])
C_hh = float(calib['C_hh'])
mpc_avg = float(calib['mpc_avg'])
asset_resid = float(calib['asset_market_residual'])

print('Loaded all inputs.')

Loaded all inputs.


## Build the comparison rows

In [2]:
# Compute the empirical 1-step autocorrelation of the discretised Markov chain
log_e = np.log(markov['ex'])
exinv = markov['exinv']
mu = exinv @ log_e
var = exinv @ (log_e - mu) ** 2
cov_1step = exinv @ ((log_e - mu) * (markov['piex'] @ (log_e - mu)))
implied_rho = cov_1step / var
implied_sigma_cross = float(np.sqrt(var))

rows = [
    # (Group, Parameter, Paper value, MATLAB value, Python value, Source)
    ('Preferences', '$\\sigma$ (CRRA)', '1.0', '1.0', f'{gamma:.3f}', 'paper §4.1.1; Main.m:32'),
    ('Preferences', '$\\theta$ (intratemporal EOS)', '1.0', '1.0', f'{theta:.3f}', 'paper §4.1.1; Main.m:33'),
    ('Preferences', '$\\chi = 1 - \\omega$ (home-goods share)', '0.6', '0.6', f'{1 - omega:.3f}', 'paper §4.1.1; Main.m:35'),
    ('Preferences', '$\\varphi$ (Frisch)', '0.5 (paper Table 1 footnote)', '0.5', f'{varphi:.3f}', 'paper §4.1.1; Main.m:39'),
    ('Preferences', '$\\beta$ (discount factor)', '0.983', '0.9832 (warm start) → calibrated', f'{beta_star:.5f}', 'paper §4.1.1; Main.m:215, calibrate_beta_open_tg.m'),
    ('Productivity', '$\\rho_l$ (AR(1) persistence)', '0.97', '0.97', f'{rho_l:.3f}', 'paper §4.1.2; Main.m:48'),
    ('Productivity', '$\\sigma_l$ (cross-sectional std)', '0.84', '0.84', f'{sigma_cross:.3f}', 'paper §4.1.2; Main.m:50'),
    ('Productivity', 'Implied 1-step autocorr (verification)', '0.97 (target)', '0.97', f'{implied_rho:.6f}', 'notebook 01 (Rouwenhorst exactness)'),
    ('Productivity', 'Implied cross-sectional std (verification)', '0.84 (target)', '0.84', f'{implied_sigma_cross:.6f}', 'notebook 01'),
    ('Technology', '$\\alpha$ (capital share)', '0.33', '0.33', f'{alpha:.3f}', 'paper §4.1.3; Main.m:31'),
    ('Technology', '$\\delta$ (quarterly depreciation)', '0.02', '0.02 (= 0.08/4)', f'{delta:.4f}', 'paper §4.1.3; Main.m:34'),
    ('Technology', '$K/Y$ (quarterly capital-output ratio)', '9.70', '9.703 (= cap_gdp)', f'{K_implied / y_ss:.4f}', 'paper §4.1.3; Main.m:64, derived'),
    ('Foreign sector', '$\\bar B / Y$ (NFA / quarterly GDP)', '-2.0 (paper text §4.1.4)', '-1.418 (per-capita MATLAB)', f'{b_agg_ss / y_ss:.4f}', 'paper §4.1.4; Main.m:19, 79'),
    ('Foreign sector', '$\\theta^{*}$ (foreign demand elasticity)', '3.0 (FLOR 2018)', 'computed endogenously per Main.m:254', '— (Phase C)', 'paper §4.1.4; Main.m:254'),
    ('Nominal rigidities', '$\\varepsilon$ (intra-domestic EOS)', '10.0', '10.0', f'{epsilon_f:.1f}', 'paper §4.1.5; Main.m:40-41'),
    ('Nominal rigidities', '$\\zeta$ (Rotemberg adj cost)', '100.0', '100.0', f'{theta_nr:.1f}', 'paper §4.1.5; Main.m:42'),
    ('Nominal rigidities', 'NKPC slope ($\\varepsilon/\\zeta$)', '0.10', '0.10', f'{epsilon_f / theta_nr:.3f}', 'paper §4.1.5; derived'),
    ('— Derived equilibrium quantities —', '', '', '', '', ''),
    ('Steady state', '$r_{ss}$ (quarterly real rate)', '~1.0% / quarter', '0.01061', f'{r_ss:.5f}', 'derived from Main.m:71'),
    ('Steady state', '$r_{ss}$ annualised', '~4% / year', '~4.32%', f'{(1 + r_ss) ** 4 - 1:.4%}', 'derived'),
    ('Steady state', '$A^{HH}$ (asset demand)', '$K + B$', '$K + B$', f'{A_hh:.4f}', 'notebook 06 (eigenvector of T)'),
    ('Steady state', 'Asset-market residual', '0.0 (calibrated)', '~0.0', f'{asset_resid:+.2e}', 'notebook 07 (brentq)'),
    ('Steady state', 'Aggregate MPC (quarterly)', '— (not in paper Table 1)', '0.109', f'{mpc_avg:.4f}', 'notebook 06 (PCHIP derivative of $c(a)$)'),
]

for r in rows:
    print(r)

('Preferences', '$\\sigma$ (CRRA)', '1.0', '1.0', '1.000', 'paper §4.1.1; Main.m:32')
('Preferences', '$\\theta$ (intratemporal EOS)', '1.0', '1.0', '1.000', 'paper §4.1.1; Main.m:33')
('Preferences', '$\\chi = 1 - \\omega$ (home-goods share)', '0.6', '0.6', '0.600', 'paper §4.1.1; Main.m:35')
('Preferences', '$\\varphi$ (Frisch)', '0.5 (paper Table 1 footnote)', '0.5', '0.500', 'paper §4.1.1; Main.m:39')
('Preferences', '$\\beta$ (discount factor)', '0.983', '0.9832 (warm start) → calibrated', '0.98322', 'paper §4.1.1; Main.m:215, calibrate_beta_open_tg.m')
('Productivity', '$\\rho_l$ (AR(1) persistence)', '0.97', '0.97', '0.970', 'paper §4.1.2; Main.m:48')
('Productivity', '$\\sigma_l$ (cross-sectional std)', '0.84', '0.84', '0.840', 'paper §4.1.2; Main.m:50')
('Productivity', 'Implied 1-step autocorr (verification)', '0.97 (target)', '0.97', '0.970000', 'notebook 01 (Rouwenhorst exactness)')
('Productivity', 'Implied cross-sectional std (verification)', '0.84 (target)', '0.84', '0.8

## Render as a markdown table for `verification.md`

In [3]:
lines = []
lines.append('# Verification table\n')
lines.append('Cross-check of de Ferra–Mitman–Romei (2020) Table 1 against our independent Python pipeline.\n')
lines.append('')
lines.append('| Group | Parameter / Quantity | Paper | MATLAB | **Python** | Source |')
lines.append('|---|---|---|---|---|---|')
for group, name, paper, mat, py, src in rows:
    if name == '':  # group separator
        lines.append(f'| **{group}** | | | | | |')
        continue
    lines.append(f'| {group} | {name} | {paper} | {mat} | **{py}** | {src} |')

lines.append('')
lines.append('## Notes')
lines.append('')
lines.append('- **`β`**: paper Table 1 reports 0.983; MATLAB uses 0.9832 as a warm-start guess (`Main.m` line 203) and then runs `fsolve` over `calibrate_beta_open_tg.m` (line 215). Our Python brentq calibration converges to `β* = ' + f'{beta_star:.5f}' + '`, which differs from 0.983 by `' + f'{abs(beta_star - 0.983):.5f}' + '`.')
lines.append('- **`K/Y`**: target value 9.703 from `target.cap_gdp` (Main.m line 21); our calibrated SS produces ' + f'{K_implied / y_ss:.4f}' + ' to within numerical tolerance.')
lines.append('- **`NFA/Y`**: paper text §4.1.4 quotes `-2.0` but the MATLAB calibration computes `-1.418` from per-capita Eurostat / HFCS numbers (Main.m lines 18-19, 22). We use the MATLAB value verbatim.')
lines.append('- **`θ*` (foreign demand elasticity)**: paper text quotes 3.0; the MATLAB *computes* `theta_star` endogenously in `Main.m` line 254 to clear an asset-market identity. This row will be filled in by Phase C when we port the transition system.')
lines.append('- **MPC**: not a calibration target in the paper; reported here as a model-implied moment (~11% per quarter / ~37% annual).')
lines.append('')

md = '\n'.join(lines)
out_path = OUTPUT_DIR / 'verification_table.md'
out_path.write_text(md)
print(f'Wrote {out_path.resolve()}\n')
print(md)

Wrote /Users/siyingli/github/de-Ferra2020-kz/Code/Python/output/verification_table.md

# Verification table

Cross-check of de Ferra–Mitman–Romei (2020) Table 1 against our independent Python pipeline.


| Group | Parameter / Quantity | Paper | MATLAB | **Python** | Source |
|---|---|---|---|---|---|
| Preferences | $\sigma$ (CRRA) | 1.0 | 1.0 | **1.000** | paper §4.1.1; Main.m:32 |
| Preferences | $\theta$ (intratemporal EOS) | 1.0 | 1.0 | **1.000** | paper §4.1.1; Main.m:33 |
| Preferences | $\chi = 1 - \omega$ (home-goods share) | 0.6 | 0.6 | **0.600** | paper §4.1.1; Main.m:35 |
| Preferences | $\varphi$ (Frisch) | 0.5 (paper Table 1 footnote) | 0.5 | **0.500** | paper §4.1.1; Main.m:39 |
| Preferences | $\beta$ (discount factor) | 0.983 | 0.9832 (warm start) → calibrated | **0.98322** | paper §4.1.1; Main.m:215, calibrate_beta_open_tg.m |
| Productivity | $\rho_l$ (AR(1) persistence) | 0.97 | 0.97 | **0.970** | paper §4.1.2; Main.m:48 |
| Productivity | $\sigma_l$ (cross-sectional

## JSON dump of all calibrated values

For programmatic access (e.g. by `reproduce.sh` or downstream verification scripts).

In [4]:
summary = {
    'paper_table1': {
        'alpha': 0.33,
        'delta': 0.02,
        'beta': 0.983,
        'omega': 0.4,
        'sigma': 1.0,
        'theta': 1.0,
        'varphi': 0.5,
        'epsilon': 10.0,
        'zeta_rotemberg': 100.0,
        'rho_l': 0.97,
        'sigma_l': 0.84,
        'K_over_Y': 9.70,
        'NFA_over_Y_text': -2.0,
    },
    'matlab_replication_package': {
        'beta_warmstart': 0.9832,
        'NFA_over_Y_computed': -1.418,
        'cap_gdp_target': float(p['target__cap_gdp']),
        'nfa_gdp_target': float(p['target__nfa_gdp']),
    },
    'python_calibrated': {
        'beta_calibrated': beta_star,
        'K_implied': K_implied,
        'K_over_Y_python': K_implied / y_ss,
        'A_hh': A_hh,
        'C_hh': C_hh,
        'mpc_avg': mpc_avg,
        'asset_market_residual': asset_resid,
        'r_ss': r_ss,
        'r_ss_annual': (1 + r_ss) ** 4 - 1,
        'verified_rho': float(implied_rho),
        'verified_sigma_cross': implied_sigma_cross,
    },
}

out_path = OUTPUT_DIR / 'calibration_summary.json'
out_path.write_text(json.dumps(summary, indent=2))
print(f'Wrote {out_path.resolve()}')
print(json.dumps(summary, indent=2))

Wrote /Users/siyingli/github/de-Ferra2020-kz/Code/Python/output/calibration_summary.json
{
  "paper_table1": {
    "alpha": 0.33,
    "delta": 0.02,
    "beta": 0.983,
    "omega": 0.4,
    "sigma": 1.0,
    "theta": 1.0,
    "varphi": 0.5,
    "epsilon": 10.0,
    "zeta_rotemberg": 100.0,
    "rho_l": 0.97,
    "sigma_l": 0.84,
    "K_over_Y": 9.7,
    "NFA_over_Y_text": -2.0
  },
  "matlab_replication_package": {
    "beta_warmstart": 0.9832,
    "NFA_over_Y_computed": -1.418,
    "cap_gdp_target": 9.702766798418972,
    "nfa_gdp_target": -1.418181818181818
  },
  "python_calibrated": {
    "beta_calibrated": 0.9832249484431147,
    "K_implied": 29.715343787700355,
    "K_over_Y_python": 9.702817323049672,
    "A_hh": 25.372093896699294,
    "C_hh": 1.4532988419610977,
    "mpc_avg": 0.10866849728705862,
    "asset_market_residual": -0.00015473410669741838,
    "r_ss": 0.010609825647710614,
    "r_ss_annual": 0.04311950298853717,
    "verified_rho": 0.97,
    "verified_sigma_cross": 